In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Same pipeline as generate_pml_pr10.ipynb, but for PaDDAH (19743_1, apo; manuscript Fig. 4k). The residues in the
three high-complexity / high-dynamics cells

    complexity_bin=2, flex_bin=3
    complexity_bin=3, flex_bin=3
    complexity_bin=3, flex_bin=2

are projected onto one structure (one pml, union of the three cells).

Step 1: write 19743_1_acerr_bfactor.pdb (error from raw PRED/GT data, structure =
predicted structure in structures_split_filtered/test).
Step 2: write render_19743_1_hotcells.pml highlighting the residues in the three cells.
"""
import json
import os

import numpy as np
import pandas as pd

DATA_JSON = ("../../../data/analysis/Fig4/per_protein_analysis/ss_error_attribution_data_panav10seedens.json")
PRED_PATH = ("../../../data/analysis/Fig4/results/predictions_N_seed_ensemble_10seed.json")
GT_PATH = ("../../../data/rBMRB/cluster_30p/PANAV/test.json")
STRUCT_DIR = ("../../../data/rBMRB/cluster_30p/structures_split_filtered/test")
OUT_DIR = "."  # pml and B-factor PDBs are written next to this notebook
N_BINS = 4
SENTINEL = -1.00

# target protein and the (complexity_bin, flex_bin) cells to project
ENTRIES = {"19743_1": "PaDDAH (apo)"}
CELLS = [(2, 3), (3, 3), (3, 2)]

PML_TEMPLATE = """bg_color white
set ray_opaque_background, 0
set orthoscopic, on

set specular, 0
set reflect, 0
set ambient, 1
set depth_cue, 0
set ray_trace_mode, 1
set ray_trace_depth_factor, 1
set ray_trace_disco_factor, 1
set ray_trace_gain, 0.005
set ray_shadow, off
set antialias, 2
set cartoon_oval_length, 1
set cartoon_rect_length, 1

load {pdb_filename}, daf
hide everything, daf
show cartoon, daf
select daf_nodata, daf and b < -0.5
select daf_hasdata, daf and b > -0.5
color grey70, daf_nodata
spectrum b, white_red, daf_hasdata, minimum=0.0, maximum=5

# residues from {entry} ({name}) that fall in {cell_label}
select enriched_cell, daf and resi {resi_sel}
show sticks, enriched_cell and not name C+N+O
set stick_radius, 0.35, enriched_cell
show spheres, enriched_cell and name CA
hide everything, hydro
set sphere_scale, 0.35, enriched_cell and name CA
color yellow, enriched_cell and name CA

orient daf
#ray 1800, 1500
#png {entry}_error_on_structure.png, dpi=300
"""


def in_gate(x, r):
    return x is not None and r != "P"


def error_by_resnum(entry, pred_json, gt_json, key_of):
    gk = key_of[entry]
    seq = gt_json["sequence"][gk]
    Nv = gt_json["N"][gk]
    L = len(seq)
    pred_arr = np.array(pred_json["mean"][entry])[:L].astype(float)
    pred_mask = np.array(pred_json["mask"][entry], dtype=bool)[:L]
    return {i + 1: abs(pred_arr[i] - Nv[i])
            for i in range(L) if in_gate(Nv[i], seq[i]) and pred_mask[i]}


def write_bfactor_pdb(entry, err_by_resnum, out_path):
    struct_path = os.path.join(STRUCT_DIR, f"{entry}.pdb")
    with open(struct_path) as fh:
        lines = fh.readlines()
    out_lines = []
    n_written, n_sentinel = 0, 0
    for line in lines:
        if line.startswith(("ATOM", "HETATM")):
            resnum = int(line[22:26])
            val = err_by_resnum.get(resnum, SENTINEL)
            n_sentinel += (val == SENTINEL)
            n_written += (val != SENTINEL)
            line = line[:60] + f"{val:6.2f}" + line[66:]
        out_lines.append(line)
    with open(out_path, "w") as fh:
        fh.writelines(out_lines)
    return n_written, n_sentinel


def residues_in_cell(base, entry, complexity_bin, flex_bin):
    cell_df = base[
        (base["entry"] == entry)
        & (base["complexity_bin"] == complexity_bin)
        & (base["flex_bin"] == flex_bin)
    ]
    return sorted(cell_df["resnum"].tolist())


def main():
    os.makedirs(OUT_DIR, exist_ok=True)

    data = pd.DataFrame(json.load(open(DATA_JSON)))
    base = data.dropna(subset=["pack_N8", "s2_ired"]).copy()
    _, cx_edges = pd.qcut(base["pack_N8"], N_BINS, labels=False, retbins=True)
    _, dy_edges = pd.qcut(base["s2_ired"], N_BINS, labels=False, retbins=True)
    base["complexity_bin"] = pd.cut(base["pack_N8"], cx_edges, labels=False, include_lowest=True)
    dynamics_bin = pd.cut(base["s2_ired"], dy_edges, labels=False, include_lowest=True)
    base["flex_bin"] = (N_BINS - 1) - dynamics_bin

    pred_json = json.load(open(PRED_PATH))
    gt_json = json.load(open(GT_PATH))
    key_of = {v: k for k, v in gt_json["entry_ID"].items()}

    # ---- step 1: one bfactor pdb per protein ----
    entries = sorted(ENTRIES)
    for entry in entries:
        err_by_resnum = error_by_resnum(entry, pred_json, gt_json, key_of)
        out_path = os.path.join(OUT_DIR, f"{entry}_acerr_bfactor.pdb")
        n_written, n_sentinel = write_bfactor_pdb(entry, err_by_resnum, out_path)
        print(f"{entry}: wrote {os.path.basename(out_path)} "
              f"({n_written} atoms with real error, {n_sentinel} sentinel)")

    # ---- vmax from the raw error of the target protein ----
    all_errors = [v for entry in entries for v in error_by_resnum(entry, pred_json, gt_json, key_of).values()]
    vmax = round(float(np.quantile(all_errors, 0.98)), 2)
    print(f"\nvmax (98th pct across {len(entries)} proteins) = {vmax}")

    # ---- step 2: one pml per protein, union of the residues in the three cells ----
    cell_label = " + ".join(f"(complexity_bin={c}, flex_bin={f})" for c, f in CELLS)
    for entry in entries:
        per_cell = {(c, f): residues_in_cell(base, entry, c, f) for c, f in CELLS}
        resnums = sorted({r for rs in per_cell.values() for r in rs})
        pml_text = PML_TEMPLATE.format(
            pdb_filename=f"{entry}_acerr_bfactor.pdb",
            vmax=vmax,
            cell_label=cell_label,
            resi_sel="+".join(str(r) for r in resnums),
            entry=entry,
            name=ENTRIES[entry],
        )
        out_path = os.path.join(OUT_DIR, f"render_{entry}_hotcells.pml")
        with open(out_path, "w") as fh:
            fh.write(pml_text)
        print(f"wrote {os.path.basename(out_path)}  ({len(resnums)} highlighted residues)")
        for (c, f), rs in per_cell.items():
            print(f"    cbin{c}_fbin{f}: {rs}")

if __name__ == "__main__":
    main()